# 🍳 cAIuldron - AI Recipe Generator

**Transform ingredient photos into delicious recipes!**

## Features:
- 🔍 Multi-ingredient detection (CLIP + DETR)
- 🥗 Nutrition estimation (525+ ingredients)
- 🍳 Multi-model recipe generation (GPT-2, Llama 1B, Llama 8B)
- ✅ Time generation with format validation
- 🌐 Beautiful Gradio web interface
- 💯 100% local, no API costs

## Architecture:
This notebook integrates all modules:
1. `1_model_loading.ipynb` - Model loading
2. `2_ingredient_detection.ipynb` - Ingredient detection
3. `3_nutrition_estimation.ipynb` - Nutrition estimation
4. `4_recipe_generation.ipynb` - Recipe generation
5. `app.ipynb` (this file) - Gradio interface

## 1. Load All Modules

In [1]:
import warnings
warnings.filterwarnings('ignore')

print("Loading modules...")
print("="*60)

Loading modules...


In [ ]:
# Load Module 1: Model Loading
print("\n[1/4] Loading model loading module...")
%run 1_model_loading.ipynb


[1/4] Loading model loading module...


In [ ]:
# Load Module 2: Ingredient Detection
print("\n[2/4] Loading ingredient detection module...")
%run 2_ingredient_detection.ipynb


[2/4] Loading ingredient detection module...
✓ Imports loaded
✓ Data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data
✓ Detection mode: multi
✓ Loaded 528 ingredients
✓ CLIP model loaded


Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✓ DETR model loaded (multi-ingredient detection)
✓ Single-ingredient detection function defined
✓ Multi-ingredient detection function defined
✓ Consolidation function defined
Found: Chicken breast and Pork kidney
Primary: Chicken breast (confidence: 45.1%)


In [ ]:
# Load Module 3: Nutrition Estimation
print("\n[3/4] Loading nutrition estimation module...")
%run 3_nutrition_estimation.ipynb


[3/4] Loading nutrition estimation module...
✓ Imports loaded
✓ Data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data
✓ Nutrition database loaded: 525 ingredients
✓ Typical weights loaded
✓ Nutrition estimation function defined
Weight: 103.6g
Servings: 1
Calories per serving: 272.0 kcal


In [ ]:
# Load Module 4: Recipe Generation
print("\n[4/4] Loading recipe generation module...")
%run 4_recipe_generation.ipynb


[4/4] Loading recipe generation module...
✓ Imports loaded
✓ Llama recipe generation function defined
✓ LLM time estimation function defined
✓ GGUF time estimation function defined
✓ Format validation function defined
✓ Time field enforcement function defined
✓ Llama output parsing function defined
✓ Llama recipe generation wrapper defined
✓ GGUF output parsing function defined
✓ GGUF recipe generation function defined
✓ GPT-2 recipe generation function defined (simplified)
✓ Model router defined
✓ Diverse prompts generator defined


In [ ]:
print("\n" + "="*60)
print("✓ All modules loaded successfully!")
print("="*60)


✓ All modules loaded successfully!


## 2. Setup Gradio Interface

In [ ]:
import gradio as gr
from pathlib import Path
import time
import json

# Setup paths
PROJECT_ROOT = Path.cwd().parent.parent.parent
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = DATA_DIR / "results" / "pipeline_output"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Number of recipes to generate
NUM_RECIPES = 5

print("✓ Gradio setup complete")

✓ Gradio setup complete


## 3. Gradio Functions

In [ ]:
def gradio_detect_ingredients(image, confidence_threshold):
    """Stage 1: Detect ingredients and calculate nutrition"""
    if image is None:
        return "⚠️ Please upload an image first.", "", "{}"
    
    # Save temporary image
    temp_path = RESULTS_DIR / "temp_upload.jpg"
    image.save(temp_path)
    
    start_time = time.time()
    
    try:
        # Detect ingredients
        detected = detect_multiple_ingredients_clip(
            str(temp_path),
            object_threshold=OBJECT_DETECTION_THRESHOLD,
            ingredient_threshold=confidence_threshold or INGREDIENT_CONFIDENCE_THRESHOLD
        )
        
        if not detected:
            return "❌ No ingredients detected", "", "{}"
        
        # Consolidate detections
        result = consolidate_detections(detected)
        
        # Calculate nutrition
        estimated_width = int(result['primary_area'] ** 0.5)
        estimated_height = int(result['primary_area'] ** 0.5)
        nutrition = estimate_nutrition(
            result['primary_ingredient'],
            estimated_width,
            estimated_height
        )
        
        elapsed_time = time.time() - start_time
        
        # Format detection results
        detection_result = f"""# 🔍 Detection Results

**Ingredients**: {result['combined_ingredient']}
**Primary**: {result['primary_ingredient']}
**Confidence**: {result['primary_confidence']:.1%}
**Time**: {elapsed_time:.2f}s

✅ Detection complete! Click 'Generate Recipes' to continue.
"""
        
        # Format nutrition results
        if nutrition['success']:
            nutrition_result = f"""# 🥗 Nutrition Information

**Ingredient**: {result['primary_ingredient']}
**Weight**: {nutrition['weight_g']}g
**Servings**: {nutrition['servings']}

### Per Serving ({nutrition['per_serving']['weight_g']}g)
- Calories: {nutrition['per_serving']['calories']:.0f} kcal
- Protein: {nutrition['per_serving']['protein_g']}g
- Fat: {nutrition['per_serving']['fat_g']}g
- Carbs: {nutrition['per_serving']['carbs_g']}g
"""
        else:
            nutrition_result = "⚠️ Nutrition data not available"
        
        # Prepare data for next stage
        detection_data = {
            'combined_ingredient': result['combined_ingredient'],
            'unique_ingredients': result['unique_ingredients'],
            'primary_name': result['primary_ingredient'],
            'nutrition': nutrition if nutrition['success'] else None
        }
        
        return detection_result, nutrition_result, json.dumps(detection_data)
        
    except Exception as e:
        return f"❌ Detection failed: {e}", "", "{}"

print("✓ Detection function defined")

✓ Detection function defined


In [ ]:
def gradio_generate_recipes(detection_data_json, selected_model):
    """
    Stage 2: Generate recipes with time validation
    Note: Removed JSON saving - only display results
    """
    if not detection_data_json or detection_data_json == "{}":
        yield "⚠️ Please detect ingredients first"
        return
    
    try:
        detection_data = json.loads(detection_data_json)
    except:
        yield "❌ Invalid detection data"
        return
    
    combined_ingredient = detection_data['combined_ingredient']
    nutrition_data = detection_data.get('nutrition')
    
    # Parse selected model
    model_type = RecipeModelType.LLAMA_1B
    if "GPT-2" in selected_model:
        model_type = RecipeModelType.GPT2
    elif "Llama 3.2 1B" in selected_model:
        model_type = RecipeModelType.LLAMA_1B
    elif "Llama 3.1 8B" in selected_model:
        model_type = RecipeModelType.LLAMA_8B_GGUF
    
    yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}

⏳ Generating {NUM_RECIPES} recipes with validated time fields...
"""
    
    # Generate diverse recipes
    prompts = generate_diverse_prompts(combined_ingredient, NUM_RECIPES)
    recipes = []
    
    for idx, prompt in enumerate(prompts, 1):
        recipe = generate_recipe_with_selected_model(
            ingredient=combined_ingredient,
            cuisine=prompt['cuisine'],
            difficulty=prompt['difficulty'],
            model_type=model_type,
            nutrition_data=nutrition_data
        )
        
        # Add nutrition info
        if detection_data.get('nutrition'):
            recipe['nutrition'] = detection_data['nutrition'].get('per_serving', {})
        
        recipes.append(recipe)
        
        yield f"""# 🍳 Generating Recipes

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}

⏳ Generated {idx}/{NUM_RECIPES} recipes...
"""
    
    # Format final results
    final_result = f"""# 🍳 Generated Recipes ({len(recipes)})

**Ingredients**: {combined_ingredient}
**Model**: {selected_model}

---

"""
    
    for idx, recipe in enumerate(recipes, 1):
        # Recipe header
        final_result += f"""## {idx}. {recipe['recipe_title']}

**Cuisine**: {recipe.get('cuisine', 'N/A')} | **Difficulty**: {recipe.get('difficulty', 'N/A')}

"""
        
        # Recipe content (includes Prep Time, Cook Time, Total Time, Servings)
        if recipe.get('raw_markdown'):
            raw_md = recipe['raw_markdown']
            lines = raw_md.split('\n')
            content_lines = []
            skip_first_heading = False
            
            for line in lines:
                if not skip_first_heading and line.startswith('#') and not line.startswith('##'):
                    skip_first_heading = True
                    continue
                content_lines.append(line)
            
            cleaned_markdown = '\n'.join(content_lines)
            final_result += cleaned_markdown + "\n\n"
        else:
            # GPT-2 format (basic)
            final_result += f"*Recipe generated by {selected_model}*\n\n"
        
        final_result += "---\n\n"
    
    final_result += f"""\n✅ All recipes generated successfully!

💡 **Features Applied**:
- ✅ Time fields validated (Prep Time, Cook Time, Total Time)
- ✅ Smart servings extraction from nutrition data
- ✅ Format enforcement with fallback estimation
"""
    
    yield final_result

print("✓ Recipe generation function defined")

✓ Recipe generation function defined


## 4. Launch Gradio Interface

In [ ]:
with gr.Blocks(title="cAIuldron - AI Recipe Generator", theme=gr.themes.Soft()) as app:
    gr.Markdown("""
    # 🍳 cAIuldron - AI Recipe Generator
    
    Transform ingredient photos into delicious recipes!
    
    **100% Free • Runs Locally • Multi-Ingredient Detection • Time Validation**
    """)
    
    detection_data_state = gr.State(value="{}")
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📤 Upload Photo")
            image_input = gr.Image(type="pil", label="Ingredient Photo")
            
            gr.Markdown("### ⚙️ Detection Settings")
            confidence_slider = gr.Slider(
                minimum=0.05,
                maximum=0.50,
                value=0.15,
                step=0.01,
                label="Confidence Threshold"
            )
            
            detect_btn = gr.Button("🔍 Detect Ingredients", variant="primary", size="lg")
            
            gr.Markdown("### 🤖 Recipe Generation Model")
            model_selector = gr.Dropdown(
                choices=[
                    "GPT-2 (Fast)",
                    "Llama 3.2 1B (Recommended)",
                    "Llama 3.1 8B (Best Quality)"
                ],
                value="Llama 3.2 1B (Recommended)",
                label="Select Model"
            )
            
            generate_btn = gr.Button("🍳 Generate Recipes", variant="secondary", size="lg")
            
            gr.Markdown("""
            ### 💡 Usage:
            1. **Upload** - Upload ingredient photo
            2. **Detect** (~1s) - Detect ingredients & nutrition
            3. **Select Model** - Choose generation model
            4. **Generate** (~30-60s) - Get 5 recipes
            
            ### 🤖 Model Comparison:
            - **GPT-2**: Fast, good quality
            - **Llama 1B**: Recommended ⭐
            - **Llama 8B**: Best quality, slower
            """)
        
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Results")
            
            with gr.Tabs():
                with gr.Tab("🔍 Detection"):
                    detection_output = gr.Markdown(value="Upload an image and click 'Detect Ingredients'")
                
                with gr.Tab("🥗 Nutrition"):
                    nutrition_output = gr.Markdown(value="Nutrition info will appear after detection")
                
                with gr.Tab("🍳 Recipes"):
                    recipes_output = gr.Markdown(value="Select a model and click 'Generate Recipes'")
    
    gr.Markdown("""
    ---
    **Powered by:** 
    - 🔍 CLIP + DETR (Ingredient Detection)
    - 🤖 GPT-2 / Llama 3.2 1B / Llama 3.1 8B (Recipe Generation)
    - 🥗 USDA FoodData Central (Nutrition Database)
    - ✅ Format Validation with LLM-based Fallback
    
    **Modular Architecture:**
    - `1_model_loading.ipynb` - Model management
    - `2_ingredient_detection.ipynb` - CLIP + DETR detection
    - `3_nutrition_estimation.ipynb` - Nutrition calculation
    - `4_recipe_generation.ipynb` - Recipe generation logic
    - `app.ipynb` - This Gradio interface
    """)
    
    # Connect buttons
    detect_btn.click(
        fn=gradio_detect_ingredients,
        inputs=[image_input, confidence_slider],
        outputs=[detection_output, nutrition_output, detection_data_state]
    )
    
    generate_btn.click(
        fn=gradio_generate_recipes,
        inputs=[detection_data_state, model_selector],
        outputs=[recipes_output]
    )

print("✓ Gradio interface created")

TypeError: BlockContext.__init__() got an unexpected keyword argument 'language'

## 5. Launch Application

In [ ]:
print("\n" + "="*80)
print("🚀 LAUNCHING cAIuldron - AI Recipe Generator")
print("="*80)
print("\nYour Trained Models:")
print("  ✅ GPT-2 Fine-tuned (Fast)")
print("  ✅ Llama 3.2 1B Fine-tuned with LoRA (Recommended)")
print("  📦 Llama 3.1 8B GGUF (Base model, best quality)")
print("\nFeatures:")
print("  🔍 Multi-ingredient detection (CLIP + DETR)")
print("  🤖 3 model options (2 are your trained models)")
print("  🥗 Nutrition estimation (525+ ingredients)")
print("  ✅ Time validation with format enforcement")
print("  🎯 Modular architecture (easy to maintain)")
print("\nInterface: http://127.0.0.1:7861")
print("To stop: Press Stop button or Kernel → Interrupt")
print("="*80 + "\n")

app.launch(
    inbrowser=True,
    server_port=7860,
    share=False
)

---

## 📝 Notes

### Modular Architecture Benefits:
1. **Easy Maintenance** - Each module can be updated independently
2. **Clear Separation** - Each file has a single responsibility
3. **Reusability** - Modules can be imported in other projects
4. **Testing** - Each module can be tested separately
5. **Readability** - Smaller, focused files are easier to understand

### Changes from Original:
- ✅ **Removed**: JSON file saving (simplified)
- ✅ **Removed**: Auto-fix functions (not needed)
- ✅ **Removed**: Redundant GPT-2 parsing (simplified)
- ✅ **Kept**: Core detection, nutrition, and generation logic
- ✅ **Kept**: Time validation and format enforcement
- ✅ **Improved**: Modular structure for better maintenance

### File Structure:
```
FINAL/
├── 1_model_loading.ipynb          # Model management
├── 2_ingredient_detection.ipynb   # CLIP + DETR detection
├── 3_nutrition_estimation.ipynb   # Nutrition calculation
├── 4_recipe_generation.ipynb      # Recipe generation
└── app.ipynb                      # Gradio interface (this file)
```